# Data-loader benchmarking

This notebook benchmarks the `dataloader` function and its impact on results and performance. For this purpose, we tested:
1. **Model performance:** Measured the total time elapsed when training the same model on the same data.
2. **Model accuracy:** Tested whether using `dataloader` produces any significant change in predictive power.

Results path: `/home/mriveraceron/glv-research/Results/train_benchmark`

Training data path: `/home/mriveraceron/glv-research/data_tensors/gnn_proof_full`

Model performance results are shown below.

In [ ]:
# Script for styling the Jupyter Notebook using an external CSS file
from IPython.display import HTML

with open("style.css", "r") as f:
    css = f.read()

HTML(f"<style>{css}</style>")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Add the font directory directly
import matplotlib.font_manager as fm
fm.fontManager.addfont('/home/mriveraceron/fonts_git/Times New Roman.ttf')
fm.fontManager.addfont('/home/mriveraceron/fonts_git/Times New Roman Bold.ttf')
fm.fontManager.addfont('/home/mriveraceron/fonts_git/Times New Roman Italic.ttf')
fm.fontManager.addfont('/home/mriveraceron/fonts_git/Times New Roman Bold Italic.ttf')

plt.rcParams.update({
    'font.family':  'serif',
    'font.serif':   'Times New Roman',
    'font.size':    12,                  # sets the global base size
    'axes.facecolor':   'white',        # ← add this
    'figure.facecolor': 'white',        # ← and this (covers fig too)
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.color':       '#e0e0e0',
    'grid.linewidth':   0.8,
    'grid.alpha':       0.7,
})


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import matplotlib.ticker as ticker
import os
import glob
import numpy as np


# --- Module-level constants ---
_LIMS       = (-0.02, 1.05)
_TICK_MAJOR = 0.2
_TICK_MINOR = 0.05

def _style_colorbar(cb):
    cb.set_label('Recuento (escala log)', fontsize=11, labelpad=8)
    cb.ax.tick_params(labelsize=10, direction='in')
    cb.outline.set_linewidth(0.8)


def _style_ticks(ax):
    for axis in (ax.xaxis, ax.yaxis):
        axis.set_major_locator(ticker.MultipleLocator(_TICK_MAJOR))
        axis.set_minor_locator(ticker.MultipleLocator(_TICK_MINOR))
        axis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))


def corr_plt(data, method, ax, label, fig):
    # --- Data & correlations ---
    mt, mp = data['mt'], data['mp']
    r_p, _ = pearsonr(mt, mp)
    r_s, _ = spearmanr(mt, mp)
    r_p = float(r_p)
    r_s = float(r_s)
    # --- Hexbin ---
    hb = ax.hexbin(np.log1p(mp), np.log1p(mt),
                   gridsize=45, cmap='YlOrRd', mincnt=1,
                   bins='log', linewidths=0.15, edgecolors='none')
    # --- Colorbar ---
    _style_colorbar(fig.colorbar(hb, ax=ax, pad=0.03, shrink=0.82, aspect=22))
    # --- Identity line ---
    ax.plot(_LIMS, _LIMS, color='0.3', linewidth=1.2, linestyle='--', alpha=0.85, zorder=5, label='Correlación perfecta')
    # --- Axes ---
    ax.set(xlabel='Keystoneness predicho', ylabel='Keystoneness observado', xlim=_LIMS, ylim=_LIMS)
    ax.set_title(method, fontsize=11, color='0.4', pad=6)
    ax.set_xlabel('Keystoneness predicho', labelpad=8)
    ax.set_ylabel('Keystoneness observado', labelpad=8)
    _style_ticks(ax)
    # --- Correlation annotation ---
    ax.text(
        0.05, 0.95,
        f'$r$ = {r_p:.3f}  (Pearson)\n$\\rho$ = {r_s:.3f}  (Spearman)',
        transform=ax.transAxes, ha='left', va='top',
        fontsize=10, fontfamily='serif', zorder=10,
        bbox=dict(boxstyle='round,pad=0.5,rounding_size=0.4',
                  facecolor='white', edgecolor='0.75',
                  linewidth=0.8, alpha=0.9),
    )
    # --- Panel label & legend ---
    ax.text(-0.1, 1.05, label, transform=ax.transAxes,fontsize=14, fontweight='bold', va='top', ha='left')
    ax.legend(loc='lower right', fontsize=10, framealpha=0.9, edgecolor='0.75')
    ax.spines[['top', 'right']].set_visible(False)


# Script chunk for models performance
dir = '/home/mriveraceron/glv-research/Results/train_benchmark'
paths = sorted(glob.glob(f'{dir}/*.npz'))  # sorted for consistent ordering
names = ['DataLoader mezclado', 'DataLoader sin mezclar', 'orden estándar']

# Generate panel plot
plt.close('all')
plt.clf()
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(18, 5))
fig.suptitle('Figura 8: Desempeño de los modelos en la predicción de keystoneness', fontweight='bold')
id = ['A', 'B', 'C']

for data_path, method, ax, label in zip(paths, names, axes, id):
    name = os.path.basename(data_path).rsplit('.', 1)[0]
    data = np.load(data_path)
    p = corr_plt(data, method, ax, label, fig)

plt.savefig('/home/mriveraceron/glv-research/thesis_plots/DLbenchmark_predictions.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.stats import pearsonr, spearmanr, gaussian_kde

# Script chunk for models performance
dir = '/home/mriveraceron/glv-research/Results/train_benchmark'
x_data = np.load('/home/mriveraceron/glv-research/Results/train_benchmark/preds_std_ordered.npz')['mp']
y_data = np.load('/home/mriveraceron/glv-research/Results/train_benchmark/preds_dl_shuffled.npz')['mp']

# --- Correlations ---
r_p, _ = pearsonr(x_data, y_data)
r_s, _ = spearmanr(x_data, y_data)
x, y   = np.log1p(x_data), np.log1p(y_data)

# --- Figure ---
plt.clf()
LIMS_LOG = [0,1]
fig, ax = plt.subplots(figsize=(5, 5))
# Hexbin
hb = ax.hexbin(x, y, gridsize=45, cmap='YlOrRd', mincnt=1, bins='log', linewidths=0.15, edgecolors='none')
# Colorbar
cb = fig.colorbar(hb, ax=ax, pad=0.03, shrink=0.82, aspect=22)
cb.set_label('Recuento (escala log)', fontsize=11, labelpad=8)
cb.ax.tick_params(labelsize=10, direction='in')
cb.outline.set_linewidth(0.8)
# Identity line
ax.plot(LIMS_LOG, LIMS_LOG, color='0.3', linewidth=1.2, linestyle='--', alpha=0.85, zorder=5, label='Correlación perfecta')
# Labels & titles
ax.set_xlabel('orden estándar (log1p)', labelpad=8)
ax.set_ylabel('dataloader mezclado (log1p)', labelpad=8)
ax.set_xlim(LIMS_LOG)
ax.set_ylim(LIMS_LOG)
fig.suptitle('GraphConv — valores de keystoneness predichos',fontsize=13, fontfamily='serif')
ax.set_title('corrlación entre métodos', fontsize=11, color='0.4', pad=6)
# Ticks
for axis in (ax.xaxis, ax.yaxis):
    axis.set_major_locator(ticker.MultipleLocator(0.1))
    axis.set_minor_locator(ticker.MultipleLocator(0.02))
    axis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))
 

# Correlation annotation
ax.text(
    0.05, 0.95,
    f'$r$ = {r_p.item():.3f}$\;$(Pearson)\n$\\rho$ = {r_s.item():.3f}$\;$(Spearman)',
    transform=ax.transAxes, ha='left', va='top',
    fontsize=10, fontfamily='serif', zorder=10,
    bbox=dict(boxstyle='round,pad=0.5,rounding_size=0.4',
              facecolor='white', edgecolor='0.75',
              linewidth=0.8, alpha=0.9),
)
# Legend & spines
ax.legend(loc='lower right', fontsize=9, framealpha=0.9, edgecolor='0.75')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(f'{dir}/dls_st_corr.png')
plt.show()

# Interpretation

As shown in the figures above, the shuffled dataloader method and the standard method yield similar correlations; however, the sorted dataloader method is not a viable alternative for model training.

Furthermore, the shuffled dataloader method outperforms the standard method, increasing the positive predictive value by 10% while also improving correlations.

In [ ]:
# Script chunk for model loss
import glob
import numpy as np

# --- Config ---
RESULTS_DIR = '/home/mriveraceron/glv-research/Results/train_benchmark'
SAVE_PATH   = '/home/mriveraceron/glv-research/thesis_plots/DLbenchmark_loss.png'
names       = ['DataLoader mezclado', 'DataLoader sin mezclar', 'Orden estándar']
colors      = ['#2166ac', '#d6604d', '#4dac26']

# --- Data ---
paths = sorted(glob.glob(f'{RESULTS_DIR}/*.npz'))

# --- Plot ---
plt.close('all')
fig, ax = plt.subplots(figsize=(8, 5))
fig.suptitle('Figura 9: Evolución de la pérdida durante el entrenamiento', fontsize=12, fontweight='bold')

for path, name, color in zip(paths, names, colors):
    loss = np.load(path)['loss_history']
    epochs = np.arange(1, len(loss) + 1)
    ax.plot(epochs, loss, color=color, linewidth=1.8, label=name, alpha=0.9)
    ax.fill_between(epochs, loss, alpha=0.07, color=color)

# --- Axes ---
ax.set_xlabel('Época',   labelpad=8, fontsize=11)
ax.set_ylabel('Pérdida', labelpad=8, fontsize=11)

# --- Ticks ---
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
ax.tick_params(axis='both', labelsize=10, direction='in', length=4)

# --- Spines ---
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['bottom', 'left']].set_linewidth(0.8)

# --- Legend ---
ax.legend(loc='upper right', fontsize=10, framealpha=0.9, edgecolor='0.75', fancybox=True)

plt.tight_layout()
plt.savefig(SAVE_PATH, dpi=300, bbox_inches='tight')
plt.show()

# Interpretation

As shown in the figures above, all three models successfully learned from the data, as evidenced by the consistent decrease in loss across epochs. However, the standard order model exhibits the highest loss at every epoch. This is an expected outcome, since training on data in its original order introduces batch correlation bias, which produces less representative gradient estimates and slows convergence. Furthermore, the sorted dataloader artificially reduces loss by homogenizing samples within each batch, making predictions locally easier but at the cost of generalization.

The shuffled dataloader outperforms both methods because randomizing the sample order ensures each batch is representative of the full data distribution, producing less biased gradient estimates and leading to faster, more stable convergence.

In [ ]:
# Script chunk for model loss
import glob
import numpy as np

# --- Config ---
RESULTS_DIR = '/home/mriveraceron/glv-research/Results/train_benchmark'
SAVE_PATH   = '/home/mriveraceron/glv-research/thesis_plots/DLbenchmark_elapsed.png'
names       = ['DataLoader mezclado', 'DataLoader sin mezclar', 'Orden estándar']
colors      = ['#2166ac', '#d6604d', '#4dac26']
offsets = [(6, 0), (6, -12), (6, 0)]  # shift red label down

# --- Data ---
paths = sorted(glob.glob(f'{RESULTS_DIR}/*.npz'))

# --- Plot ---
plt.close('all')
fig, ax = plt.subplots(figsize=(8, 5))
fig.suptitle('Figura 10: Evolución del tiempo transcurrido por método', fontsize=12, fontweight='bold')

for (path, name, color), (dx, dy) in zip(zip(paths, names, colors), offsets):
    elapsed_history = np.cumsum(np.load(path)['epochs_history'])
    epochs = np.arange(1, len(elapsed_history) + 1)
    ax.plot(epochs, elapsed_history, color=color, linewidth=1.8, label=name, alpha=0.9)
    ax.fill_between(epochs, elapsed_history, alpha=0.07, color=color)
    # Annootate final value
    final_val = elapsed_history[-1]
    ax.annotate(
        f'{final_val:.1f}s',
        xy=(epochs[-1], final_val),
        xytext=(dx, dy),
        textcoords='offset points',
        fontsize=9, color=color, fontweight='bold', va='center'
    )

# --- Axes ---
ax.set_xlabel('Época',   labelpad=8, fontsize=11)
ax.set_ylabel('Tiempo transcurrido (segundos)', labelpad=8, fontsize=11)

# --- Ticks ---
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
ax.tick_params(axis='both', labelsize=10, direction='in', length=4)

# --- Spines ---
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['bottom', 'left']].set_linewidth(0.8)

# --- Legend ---
ax.legend(loc='upper left', fontsize=10, framealpha=0.9, edgecolor='0.75', fancybox=True)

plt.tight_layout()
plt.savefig(SAVE_PATH, dpi=300, bbox_inches='tight')
plt.show()

## Interpretation

Models were trained for 700 epochs, with elapsed training times shown in the figures above. The standard method required the most training time, exceeding the dataloader-based methods by approximately 25-fold. This is attributed to its sample-wise training procedure, in which each sample is individually fed through the model, the loss is computed, and the weights are updated accordingly — a process that is computationally expensive compared to mini-batch dataloader methods.

Across all three methods, the shuffled dataloader consistently demonstrated the best performance, making it the most suitable approach for large-scale model training.

In [ ]:
# Script for confussion matrixes

# Imports
import numpy as np
import pandas as pd
import glob 
from IPython.display import display

# Experiment directory
dir = '/home/mriveraceron/glv-research/Results/train_benchmark'
paths = sorted(glob.glob(f'{dir}/*.npz'))  # sorted for consistent ordering
names = ['DataLoader mezclado', 'DataLoader sin mezclar', 'orden estándar']

for model, path in zip(names, paths):
    data        = np.load(path)
    idxt        = data['idxt']
    idxp        = data['idxp']
    graph_nodes = data['nodes']
    # Compute mask once, reuse for tp and fp
    correct = idxt == idxp
    tp = correct.sum()
    fp = len(idxt) - tp          # avoids a second full pass over the array
    fn = fp
    tn = graph_nodes.sum() - len(graph_nodes) - fp   # vectorized sum(nodes - 1)
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    ppv      = tp / (tp + fp)
    print(f'Model {model} | Accuracy: {accuracy:.4f} | PPV: {ppv:.4f} | By chance: {1/graph_nodes.mean():.4f}')
    df_cm = pd.DataFrame(
        [[tp, fp], [fn, tn]],
        index   = ['Expected_Positive', 'Expected_Negative'],
        columns = ['Predicted_Positive', 'Predicted_Negative']
    )
    display(df_cm)